In [1]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.2 MB/s eta 0:00:00


In [2]:
import qiskit
print(qiskit.__version__)

2.5.2


Import Required Libraries

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

Exercise 1 Easy - Implement Oracle for Constant Function f(x) = 0 and Verify Output '0'

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Construct circuit for f0(x) = 0
circ_c0 = QuantumCircuit(2, 1)
# Ancilla preparation in |-> and query qubit in |+>
circ_c0.x(1)
circ_c0.h([0, 1])
circ_c0.barrier()
# Identity oracle: f(x) = 0 leaves target unchanged
# (no gate required)
circ_c0.barrier()
# Recombination via Hadamard on query register
circ_c0.h(0)
circ_c0.measure(0, 0)
backend = AerSimulator()
counts_out = backend.run(circ_c0, shots=1024).result().get_counts()
print("--- Deutsch Circuit [f(x) = 0] ---")
print(circ_c0)
print("\nMeasured Outcome Frequencies:", counts_out)
print("Classification Status:", "Constant (f(0) == f(1))" if '0' in counts_out else "Balanced")

--- Deutsch Circuit [f(x) = 0] ---
     ┌───┐      ░  ░ ┌───┐┌─┐
q_0: ┤ H ├──────░──░─┤ H ├┤M├
     ├───┤┌───┐ ░  ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░──░───────╫─
     └───┘└───┘ ░  ░       ║ 
c: 1/══════════════════════╩═
                           0 

Measured Outcome Frequencies: {'0': 1024}
Classification Status: Constant (f(0) == f(1))


Exercise 2 Medium -
Implement Oracle for Balanced Function f(x) = x and Verify Output '1'

In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Circuit construction for f(x) = x
circ_bx = QuantumCircuit(2, 1)
# Input |+> and target |->
circ_bx.x(1)
circ_bx.h([0, 1])
circ_bx.barrier()
# Oracle f(x) = x using a Controlled-NOT gate
circ_bx.cx(0, 1)
circ_bx.barrier()
# Interference and measurement
circ_bx.h(0)
circ_bx.measure(0, 0)
result_counts = backend.run(circ_bx, shots=1024).result().get_counts()
print("--- Deutsch Circuit [f(x) = x] ---")
print(circ_bx)
print("\nMeasured Outcome Frequencies:", result_counts)
print("Classification Status:", "Balanced (f(0) != f(1))" if '1' in result_counts else "Constant")

--- Deutsch Circuit [f(x) = x] ---
     ┌───┐      ░       ░ ┌───┐┌─┐
q_0: ┤ H ├──────░───■───░─┤ H ├┤M├
     ├───┤┌───┐ ░ ┌─┴─┐ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤ X ├─░───────╫─
     └───┘└───┘ ░ └───┘ ░       ║ 
c: 1/═══════════════════════════╩═
                                0 

Measured Outcome Frequencies: {'1': 1024}
Classification Status: Balanced (f(0) != f(1))


Exercise 3 Hard -
Generic Oracle-Selection Function for All Four Single-Bit Functions

In [9]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def generate_oracle(function_label):
    oracle = QuantumCircuit(2, name=function_label)
    if function_label == "const_zero":       # f(x) = 0
        pass
    elif function_label == "const_one":      # f(x) = 1
        oracle.x(1)
    elif function_label == "balanced_id":    # f(x) = x
        oracle.cx(0, 1)
    elif function_label == "balanced_inv":   # f(x) = 1 - x
        oracle.x(0)
        oracle.cx(0, 1)
        oracle.x(0)
    else:
        raise ValueError("Invalid function label")
    return oracle
def evaluate_function(fn_name):
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h([0, 1])
    qc.compose(generate_oracle(fn_name), inplace=True)
    qc.h(0)
    qc.measure(0, 0)
    counts = AerSimulator().run(qc, shots=500).result().get_counts()
    readout = list(counts.keys())[0]
    return readout, "Constant" if readout == '0' else "Balanced"
cases = ["const_zero", "const_one", "balanced_id", "balanced_inv"]
print(f"{'Oracle Target':<18}{'Measured Bit':<16}{'Evaluated Class':<18}{'Verification'}")
print("-" * 65)
for target in cases:
    bit_res, cls_res = evaluate_function(target)
    verdict = "CORRECT" if (cls_res == "Constant" and "const" in target) or \
                           (cls_res == "Balanced" and "balanced" in target) else "INCORRECT"
    print(f"{target:<18}{bit_res:<16}{cls_res:<18}{verdict}")

Oracle Target     Measured Bit    Evaluated Class   Verification
-----------------------------------------------------------------
const_zero        0               Constant          CORRECT
const_one         0               Constant          CORRECT
balanced_id       1               Balanced          CORRECT
balanced_inv      1               Balanced          CORRECT


Exercise 4 Real-world -
Phase Kickback Concept in a Binary Classification Decision Problem

In [10]:
# Conceptual diagram: Quantum phase kickback as a decision hyperplane
# A feature query vector x is tested against a hypothesis classifier h(x)
# In-class / Homogeneous pattern  -> Constant phase -> Readout 0
# Boundary crossing / Separating  -> Phase inversion -> Readout 1
flowchart = [
    "--- Quantum Phase Decision Boundary Representation ---",
    "",
    "|Input Pattern x> ----[ H ]----( Control )----[ H ]----[ Measure: Parity ]",
    "                                    |",
    "|Decision Ancilla> ---[ X, H ]---[ U_f ]--------------- (Phase Shift: (-1)^f(x))",
    "",
    "Analytical Mapping:",
    "  1. Input register evaluates candidate feature states simultaneously.",
    "  2. The classifier oracle introduces a pi phase shift solely on positive hits.",
    "  3. Interference reconstructs global hypothesis properties:",
    "     * Coherent return (|0>): Constant partition (No classification boundary).",
    "     * Orthogonal shift (|1>): Balanced partition (Separating hyperplane detected)."
]
print("\n".join(flowchart))

--- Quantum Phase Decision Boundary Representation ---

|Input Pattern x> ----[ H ]----( Control )----[ H ]----[ Measure: Parity ]
                                    |
|Decision Ancilla> ---[ X, H ]---[ U_f ]--------------- (Phase Shift: (-1)^f(x))

Analytical Mapping:
  1. Input register evaluates candidate feature states simultaneously.
  2. The classifier oracle introduces a pi phase shift solely on positive hits.
  3. Interference reconstructs global hypothesis properties:
     * Coherent return (|0>): Constant partition (No classification boundary).
     * Orthogonal shift (|1>): Balanced partition (Separating hyperplane detected).


Exercise 5 Challenge -
Gate Noise Sensitivity Analysis (Imperfect Hadamard Error Study)

In [11]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Model gate over-rotation / calibration drift on Hadamard gate via Ry(pi/2 + epsilon)
def faulty_hadamard(circuit, target_qubit, angle_drift):
    circuit.ry((np.pi / 2.0) + angle_drift, target_qubit)
perturbations = [0.0, 0.15, 0.30, 0.45, 0.60, 0.75]
sample_shots = 3000
aer_sim = AerSimulator()
print("--- Gate Infidelity Impact on Balanced Oracle Classification ---")
print(f"{'Drift (rad)':<15}{'P(1) [Success]':<18}{'P(0) [Error]':<18}{'Accuracy'}")
print("=" * 64)
for eps in perturbations:
    test_qc = QuantumCircuit(2, 1)
    test_qc.x(1)
    test_qc.h(1)
    # Faulty initial rotation on query qubit
    faulty_hadamard(test_qc, 0, eps)
    # Balanced oracle operation
    test_qc.cx(0, 1)
    # Faulty final rotation
    faulty_hadamard(test_qc, 0, eps)
    test_qc.measure(0, 0)
    out = aer_sim.run(test_qc, shots=sample_shots).result().get_counts()
    success_rate = out.get('1', 0) / sample_shots
    error_rate = out.get('0', 0) / sample_shots
    print(f"{eps:<15.2f}{success_rate:<18.4f}{error_rate:<18.4f}{success_rate * 100:.2f}%")

--- Gate Infidelity Impact on Balanced Oracle Classification ---
Drift (rad)    P(1) [Success]    P(0) [Error]      Accuracy
0.00           0.0000            1.0000            0.00%
0.15           0.0000            1.0000            0.00%
0.30           0.0000            1.0000            0.00%
0.45           0.0000            1.0000            0.00%
0.60           0.0000            1.0000            0.00%
0.75           0.0000            1.0000            0.00%
